In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [2]:
import datasets
import transformers
import sentence_transformers
import faiss
import accelerate

print("All imports successful!")

All imports successful!


In [4]:
import pandas as pd
from datasets import load_dataset

print("--- Loading PubMedQA Dataset ---")
# Using 'pqa_labeled' for a quick lightweight demonstration; swap with 'pqa_artificial' for large-scale
dataset = load_dataset("qiaojin/PubMedQA", "pqa_labeled", split="train")

# Convert to Pandas DataFrame for easier manipulation
df = pd.DataFrame(dataset)

--- Loading PubMedQA Dataset ---


In [6]:
df['clean_context'] = df['context'].apply(
    lambda x: " ".join(x['contexts']) if isinstance(x, dict) else str(x)
)


In [7]:
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer

print("--- Initializing Embedding Model ---")
# Load a lightweight, high-performance embedding model
embedding_model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')

print("--- Generating Embeddings for Contexts ---")
# Encode the clean biomedical contexts
context_embeddings = embedding_model.encode(df['clean_context'].tolist(), show_progress_bar=True)

--- Initializing Embedding Model ---


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

--- Generating Embeddings for Contexts ---


Batches:   0%|          | 0/32 [00:00<?, ?it/s]

In [8]:
# Build a FAISS Index for vector similarity matching
dimension = context_embeddings.shape[1]
faiss_index = faiss.IndexFlatL2(dimension)
faiss_index.add(np.array(context_embeddings).astype('float32'))

print(f"FAISS Index successfully built with {faiss_index.ntotal} vectors.")

FAISS Index successfully built with 1000 vectors.


In [9]:
def retrieve_context(query, top_k=2):
    """
    Given a question, search the FAISS index and return the most relevant abstracts.
    """
    # Vectorize user query
    query_embedding = embedding_model.encode([query]).astype('float32')
    
    # Search index
    distances, indices = faiss_index.search(query_embedding, top_k)
    
    # Gather retrieved documents
    retrieved_docs = []
    for idx in indices[0]:
        if idx < len(df):
            retrieved_docs.append(df.iloc[idx]['clean_context'])
            
    return " \n".join(retrieved_docs)

# Quick Test of Retriever
test_query = "Do preoperative statins reduce atrial fibrillation after coronary artery bypass grafting?"
retrieved_text = retrieve_context(test_query, top_k=1)
print(f"Query: {test_query}\n\nRetrieved Context: {retrieved_text[:300]}...")

Query: Do preoperative statins reduce atrial fibrillation after coronary artery bypass grafting?

Retrieved Context: Recent studies have demonstrated that statins have pleiotropic effects, including anti-inflammatory effects and atrial fibrillation (AF) preventive effects. The objective of this study was to assess the efficacy of preoperative statin therapy in preventing AF after coronary artery bypass grafting (C...


In [11]:
import torch
from transformers import pipeline

model_id = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

llm_pipeline = pipeline(
    "text-generation",
    model=model_id,
    torch_dtype=torch.float16,
    device_map="auto"
)

print("Model loaded successfully")

config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

Model loaded successfully


In [12]:
def ask_pubmed_bot(question):
    # 1. Retrieve domain-specific background information
    context = retrieve_context(question, top_k=2)
    
    # 2. Structure structured prompt template
    prompt = f"""You are an expert biomedical Q&A assistant. Answer the user's question accurately based only on the provided scientific contexts. If the answer cannot be inferred, state that information is insufficient.

Contexts:
{context}

Question: {question}
Answer:"""

    # 3. Generate response
    outputs = llm_pipeline(
        prompt, 
        max_new_tokens=150, 
        do_sample=True, 
        temperature=0.2, 
        top_k=50, 
        top_p=0.95
    )
    
    generated_text = outputs[0]["generated_text"]
    
    # Extract only the newly generated text trailing after our prompt
    answer = generated_text[len(prompt):].strip()
    return answer

In [13]:
# Select a sample question directly from the evaluation data or write your own
sample_question = "Is standard chemotherapy superior to molecularly targeted therapy in treating advanced NSCLC patients?"

print(f"User Question:\n{sample_question}\n")
print("Thinking... Querying Database and Generating Response...\n")

bot_response = ask_pubmed_bot(sample_question)

print("--- Bot Answer ---")
print(bot_response)

Passing `generation_config` together with generation-related arguments=({'top_k', 'top_p', 'temperature', 'do_sample', 'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=150) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


User Question:
Is standard chemotherapy superior to molecularly targeted therapy in treating advanced NSCLC patients?

Thinking... Querying Database and Generating Response...

--- Bot Answer ---
Based on the provided scientific contexts, the answer to the question is "No". The comparison between standard chemotherapy and molecularly targeted therapy in treating advanced NSCLC patients does not show a significant difference in terms of response, objective remission rate, remission duration, time to response, time to best response, time to progression, overall survival, and the modified Brunner's score. Therefore, the answer to the question is "No".
